# Dataset Preparation (Public Version)

> This notebook replaces the original Udacity-supplied dataset notebook.  
> The COCO dataset used in the original cannot be shared due to licensing restrictions.

This project was originally trained using the COCO dataset for image captioning. This public version of the notebook omits the dataset loading and visualization steps.

If you'd like to recreate the pipeline:
- Download COCO 2014 dataset from [http://cocodataset.org](http://cocodataset.org)
- Use `pycocotools` to load annotations and images
- Sample captions and images to build a dataset for model training

You may replace this notebook with your own dataset exploration code.


# Image Captioning Model – Encoder/Decoder Demo

This notebook demonstrates how to construct and use a CNN encoder and RNN decoder for image captioning. The original project was completed in Udacity's Computer Vision Nanodegree using the COCO dataset. This version includes custom code and excludes any proprietary content or datasets.

The models defined below were implemented in `model.py`, with training handled in a separate notebook.


In [ ]:
# Load model architecture
from model import EncoderCNN, DecoderRNN
import torch

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Parameters
embed_size = 256
hidden_size = 512
vocab_size = 10000  # placeholder for demonstration

# Initialize encoder and decoder
encoder = EncoderCNN(embed_size).to(device)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size).to(device)

# Fake input for demonstration
batch_size = 10
image_features = torch.randn(batch_size, 3, 224, 224).to(device)
captions = torch.randint(0, vocab_size, (batch_size, 15)).to(device)

# Encode and decode
features = encoder(image_features)
outputs = decoder(features, captions)

print('Output shape:', outputs.shape)


# Training Configuration

Below is the configuration used to train the image captioning model. This includes:
- Preprocessing transforms (ResNet expected format)
- Adam optimizer with learning rate 0.001
- 20 training epochs with batch size 64
- Loss: CrossEntropyLoss

The encoder uses a pretrained ResNet-18, where only the final embedding layer is trainable. The decoder is an LSTM network.


In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms

# === LOSS AND OPTIMIZER ===
criterion = nn.CrossEntropyLoss().to(device)
params = list(decoder.parameters()) + list(encoder.embed.parameters())  # fine-tune embed layer
optimizer = optim.Adam(params, lr=0.001)

In [ ]:
# === TRANSFORMS (used during training) ===
transform_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))
])

## Training Loop (Mock Sample)

Training was performed over 20 epochs on COCO images using batches of 64. Below is a simplified mock loop used for illustrative purposes only.

In [ ]:
import time
import numpy as np
import math

# Simulated training tracking
loss_vals = []
ppl_vals = []

for epoch in range(1, 4):  # simulate 3 epochs
    epoch_loss = np.random.uniform(3.0, 5.0)
    epoch_ppl = np.exp(epoch_loss)
    loss_vals.append(epoch_loss)
    ppl_vals.append(epoch_ppl)
    print(f"Epoch {epoch}, Loss: {epoch_loss:.4f}, Perplexity: {epoch_ppl:.4f}")

In [ ]:
# === PLOT TRAINING PROGRESS ===
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(loss_vals)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(ppl_vals)
plt.title('Training Perplexity')
plt.xlabel('Epoch')
plt.ylabel('Perplexity')

plt.tight_layout()
plt.show()

# Caption Generation – Inference Demo

After training, we evaluate the model’s ability to generate natural language descriptions from unseen images.

### Captioning Pipeline
1. A test image is passed through the **CNN encoder** (ResNet-18) to extract image features.
2. These features are input to the **LSTM decoder**, which generates a caption token by token.
3. Captions are generated using:
   - **Greedy decoding** (beam width = 1)
   - **Beam search** (beam width > 1), which helps produce more coherent sentences

 *Note: Due to dataset licensing restrictions, the actual COCO test images and trained weights are not included. Examples shown below use simulated data or screenshots from the original evaluation.*



In [ ]:
# Simulated caption output
sampled_ids = [0, 3, 42, 17, 64, 129, 18, 1]  # [<start>, 'a', 'man', 'on', 'a', 'bike', '.', <end>]
sampled_caption = ['<start>', 'a', 'man', 'on', 'a', 'bike', '.', '<end>']

print("Generated Caption:")
print(" ".join(sampled_caption))

## Inference with a Test Image (Simulated)

The example below shows how a test image would be processed through the trained model. In practice, the `image_tensor` would come from a preprocessed COCO test image.


In [ ]:
# Simulate test inference with a placeholder image tensor
image_tensor = torch.randn(1, 3, 224, 224).to(device)
features = encoder(image_tensor)

# Simulated caption output
sample_ids = [0, 3, 42, 19, 75, 18, 1]  # "<start> a man with a bike . <end>"
sample_caption = "a man with a bike."

print("Predicted caption:", sample_caption)

## Examples of Generated Captions

These screenshots show actual caption outputs produced by the model using beam search (beam width = 5). Captions were generated in the original Udacity workspace.

### Good Predictions

**Image 1 – "White Boat"**  
*“a boat is on the water with trees in the background”*  
![White Boat](white_boat.png)

**Image 2 – "Man Standing"**  
*“a man standing in front of a stone oven”*  
![Man Standing](man_standing.png)


### Challenging Cases

**Image 3 – "Giraffe Overfit"**  
*“a giraffe standing under a roof”*  
**Comment:** No giraffe is present. This likely reflects the model’s overfitting to frequent COCO caption templates.  
![Giraffe](giraffe.png)

**Image 4 – "Nokia Phone"**  
*“a person holding a nokia phone”*  
**Comment:** No phone is visible. This reflects instability in the decoder's interpretation of ambiguous features.  
![Nokia](nokia.png)


In [ ]:
# Compare captions from greedy decoding and beam search
def compare_greedy_vs_beam(n=3):
    print(f"Comparing greedy (beam=1) vs. beam search (beam=5) for {n} test images:\n")

    for i in range(n):
        orig_image, image = next(iter(data_loader))
        image_tensor = image.to(device)

        features = encoder(image_tensor)

        # Greedy decoding
        greedy_ids = decoder.sample_beam_search(
            features, 
            start_token_id=start_token_id, 
            end_token_id=end_token_id, 
            beam_width=1
        )
        greedy_caption = ' '.join([
            vocab.idx2word[idx] for idx in greedy_ids 
            if vocab.idx2word[idx] not in ['<start>', '<end>']
        ])

        # Beam search
        beam_ids = decoder.sample_beam_search(
            features, 
            start_token_id=start_token_id, 
            end_token_id=end_token_id, 
            beam_width=5
        )
        beam_caption = ' '.join([
            vocab.idx2word[idx] for idx in beam_ids 
            if vocab.idx2word[idx] not in ['<start>', '<end>']
        ])

        # Convert tensor to image
        img_np = image_tensor.cpu().squeeze(0).permute(1, 2, 0).numpy()
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)

        # Display
        plt.imshow(img_np)
        plt.axis('off')
        plt.title(f"Greedy: {greedy_caption}\nBeam: {beam_caption}")
        plt.show()

# Run the comparison
compare_greedy_vs_beam(n=3)


# Conclusion

This project demonstrates the complete workflow of building and training an image captioning model using a CNN encoder and RNN decoder. While the model shows some overfitting on a small dataset subset, it successfully learns to generate grammatically coherent captions.

**Key Learnings**:
- CNN-RNN architectures are effective for image captioning
- Beam search can improve sequence quality over greedy decoding
- Dataset balance and size are critical for diversity in captions

## Next Steps

- Improve generalization by training on more COCO data
- Use a pretrained Transformer decoder (e.g., ViT-GPT2 combo)
- Implement BLEU/CIDEr evaluation metrics
- Explore curriculum learning or attention mechanisms

# Final Notes

This notebook demonstrated how to train a CNN-RNN model to generate image captions. While greedy decoding offers simplicity, beam search often results in more natural and complete descriptions.

---

### Contact

For collaboration or more information:

- **LinkedIn:** [Crystal Ford](https://www.linkedin.com/in/crystalmford)  
- **GitHub:** [github.com/crystalmford](https://github.com/crystalmford)

Thanks for reading!
